# 08 — Tuning du seuil multi-label sur train_soundscapes

But : remplacer une règle top-k fixe par un seuil choisi à partir des labels soundscape disponibles.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = c:\Users\yeyuy\Downloads\projet_ml_audio_classique_v7_structured\projet_ml_audio_classique_v7_structured


In [2]:
import pandas as pd
import numpy as np
from src.config import SOUNDSCAPE_LABELS_CSV, RESULT_DIR
from src.soundscape import tune_threshold

labels = pd.read_csv(SOUNDSCAPE_LABELS_CSV)
pred = pd.read_csv(RESULT_DIR / 'train_soundscape_predictions_top3.csv')
thresholds = np.linspace(0.10, 0.90, 17)
res = tune_threshold(labels, pred, thresholds=thresholds)
res.to_csv(RESULT_DIR / 'threshold_tuning_soundscapes.csv', index=False)
display(res)
print('Meilleur seuil:', float(res.iloc[0]['threshold']))

,threshold,macro_f1,micro_f1,samples_f1,macro_precision,macro_recall
3,0.25,0.014613,0.047498,0.040788,0.035846,0.027342
0,0.10,0.014604,0.047467,0.040788,0.035840,0.027342
1,0.15,0.014604,0.047467,0.040788,0.035840,0.027342
2,0.20,0.014604,0.047467,0.040788,0.035840,0.027342
4,0.30,0.014418,0.047202,0.040590,0.035564,0.027130
5,0.35,0.014041,0.045967,0.039260,0.035301,0.026248
6,0.40,0.013300,0.043626,0.037151,0.035023,0.024786
7,0.45,0.010605,0.035634,0.029794,0.033390,0.021288
8,0.50,0.008458,0.027636,0.022263,0.034102,0.018791
9,0.55,0.004912,0.018630,0.014535,0.040622,0.012159


Meilleur seuil: 0.25


## Décision

les meilleures performances sont obtenues pour un seuil proche de 0.25. Au-delà de cette valeur, les scores F1 et le rappel diminuent progressivement, indiquant une perte importante de détections positives. les résultats obtenus confirment également le problème est fortement déséquilibré et multi-label.

un seuil trop élevé favorise les faux négatifs, notamment pour les espèces rares ou faiblement représentées dans les soundscapes. À l'inverse, des seuils plus faibles permettent de conserver davantage de rappel tout en maintenant une précision relativement stable.

0.25 est donc retenu pour les prédictions finales sur les données de test. Cette valeur constitue un compromis entre détection des espèces présentes et limitation des faux positifs.